# M3L4 E10 — LangGraph multiagente supervisor + Langfuse
### Modulo 3 · Lecture 4 · Construccion, pruebas y trazabilidad de agentes en produccion

---

## Que necesitas saber antes

| Modulo | Concepto | Por que lo necesitas aca |
|---|---|---|
| M3L4 E09 | StateGraph con routing condicional, nodos, mapping | El patron supervisor extiende el routing condicional |
| M3L4 E08 | CallbackHandler, config con metadata | Para tracing en Langfuse |
| M3L4 E03 | Loop de agentes como patron de falla | El supervisor previene loops con `visited_agents` |
| Python | `List[str]` para tipado de historial | `visited_agents: List[str]` en el estado |

---

## Definiciones clave

| Concepto | Definicion simple | Como aparece en este notebook |
|---|---|---|
| **Supervisor** | Nodo central que decide que agente ejecutar y controla el flujo | `supervisor_node()` detecta intent, verifica `visited_agents`, marca `done` |
| **visited_agents** | Lista de agentes que ya actuaron en la sesion actual | `state['visited_agents']` se actualiza en cada nodo agente |
| **Loop prevention** | Mecanismo que evita que un agente se ejecute dos veces | Si el agente ya esta en `visited_agents`, supervisor marca `done=True` |
| **Supervisor route** | Funcion condicional que decide si seguir o terminar | `supervisor_route()` retorna nombre de nodo o `'end'` |
| **Agente con handoff** | Nodo que devuelve el control al supervisor despues de actuar | Cada agente_node retorna control al supervisor (no a END) |

---

## Arquitectura supervisor

```
START
  |
  v
supervisor_node (routing + control de loops)
  |
  |  conditional_edges (supervisor_route)
  |
  +---> hr_agent_node  ---+  (vuelve al supervisor)
  +---> it_agent_node  ---+
  +---> finance_agent_node +
  +---> legal_agent_node --+
  |
  +---> END (done=True)
```

**Diferencia con E09:** en E09 cada agente iba directo a END. Aca los agentes vuelven al supervisor, que decide si hay que ejecutar otro agente o ya terminamos.

### Por que supervisor?

El patron supervisor permite:

- Controlar cuantos agentes se invocan
- Rastrear que agentes ya actuaron (`visited_agents`)
- Detectar loops antes de que ocurran
- Centralizar la logica de orquestacion

**Objetivo del ejercicio:** implementar el patron supervisor en LangGraph — un nodo central que decide que agente actua, con historial de agentes visitados para evitar loops.

## Paso 1 — Instalacion e imports

| Libreria | Que hace |
|---|---|
| `langfuse` | SDK de Langfuse para tracing |
| `langgraph` | StateGraph con routing condicional |
| `typing.List` | Tipo `List[str]` para `visited_agents` |

```python
!pip install -q langfuse langchain langchain-openai langgraph
```

In [ ]:
!pip install -q langfuse langchain langchain-openai langgraph
print('Instalacion completa.')

In [ ]:
import os
from getpass import getpass

os.environ['LANGFUSE_PUBLIC_KEY'] = getpass('Langfuse Public Key: ')
os.environ['LANGFUSE_SECRET_KEY'] = getpass('Langfuse Secret Key: ')
os.environ['LANGFUSE_BASE_URL']   = 'https://cloud.langfuse.com'
os.environ['OPENAI_API_KEY']      = getpass('OpenAI API Key: ')
print('OK.')

In [ ]:
from typing import List
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, END
from langfuse.langchain import CallbackHandler
print('Imports OK.')

In [ ]:
def route_query_v2(query):
    q = query.lower()
    hr_kw      = ['vacaciones','licencia','recibo','nomina','rrhh']
    it_kw      = ['vpn','error','app','laptop','wifi','login','contrasena']
    finance_kw = ['factura','pago','reembolso','gasto','cobro','comprobante','salario']
    legal_kw   = ['contrato','legal','confidencialidad','nda','acuerdo']
    det = []
    if any(w in q for w in hr_kw): det.append('hr')
    if any(w in q for w in it_kw): det.append('it')
    if any(w in q for w in finance_kw): det.append('finance')
    if any(w in q for w in legal_kw): det.append('legal')
    if len(det) > 1: return 'multi_intent'
    if len(det) == 1: return det[0]
    if len(q.split()) <= 2: return 'clarification'
    return 'general'

print('Router listo.')

## Paso 2 — Estado con historial

`SupervisorState` extiende `AgentState` (de E09) agregando `visited_agents` y `done`.

```python
class SupervisorState(TypedDict):
    query: str                    # consulta original
    intent: str                   # intent detectado por supervisor
    visited_agents: List[str]     # agentes que ya actuaron (evita loops)
    response: str                 # respuesta acumulada
    done: bool                    # flag de terminacion
```

**Flujo del estado:**

```
Inicio: {'query': '...', 'intent': '', 'visited_agents': [], 'response': '', 'done': False}
    |
supervisor: {'intent': 'hr', 'done': True/False}
    |
hr_agent: {'response': '...', 'visited_agents': ['HRAgent'], 'done': True}
    |
supervisor (vuelve): verifica done=True -> END
```

In [ ]:
class SupervisorState(TypedDict):
    query: str
    intent: str
    visited_agents: List[str]
    response: str
    done: bool

print('SupervisorState definido.')

## Paso 3 — TODO: Supervisor y agentes

### supervisor_node(state)

| Parametro | Tipo | Que es |
|---|---|---|
| `state` | `SupervisorState` | Estado actual con query y visited_agents |
| **Retorna** | `dict` | `{'intent': intent, 'done': bool}` |

Logica:
1. Detectar intent con `route_query_v2(state['query'])`
2. Si intent es `'general'`, `'clarification'`, o el agente ya fue visitado: marcar `done=True`
3. Si no: marcar `done=False` y devolver intent

### Nodos agente (hr_agent_node, it_agent_node, etc.)

Cada nodo agente:
1. Agrega su nombre a `visited_agents`
2. Retorna respuesta
3. Marca `done=True`

```python
def hr_agent_node(state: SupervisorState) -> dict:
    return {
        'response': 'HRAgent: ...',
        'visited_agents': state['visited_agents'] + ['HRAgent'],
        'done': True
    }
```

In [ ]:
def supervisor_node(state: SupervisorState) -> dict:
    """
    Nodo supervisor:
    - Detecta el intent
    - Si el intent es 'general'/'clarification' o el agente ya fue visitado, marca done=True
    - Retorna {'intent': intent, 'done': bool}
    """
    # TODO
    pass

def hr_agent_node(state: SupervisorState) -> dict:
    """
    Agrega 'HRAgent' a visited_agents y retorna respuesta.
    done=True siempre (el agente cierra el turno)
    """
    # TODO
    pass

def it_agent_node(state: SupervisorState) -> dict:
    # TODO
    pass

def finance_agent_node(state: SupervisorState) -> dict:
    # TODO
    pass

def legal_agent_node(state: SupervisorState) -> dict:
    # TODO
    pass

print('Nodos definidos.')

## Paso 4 — TODO: Funcion de routing condicional del supervisor

`supervisor_route(state)` decide el siguiente paso:

- Si `done=True` -> retorna `'end'` (que mapea a END)
- Si `done=False` -> retorna el nombre del nodo segun `state['intent']`

```python
def supervisor_route(state: SupervisorState) -> str:
    if state['done']:
        return 'end'
    # mapear intent a nodo
    mapping = {'hr': 'hr_agent_node', 'it': 'it_agent_node', ...}
    return mapping.get(state['intent'], 'general_fallback')
```

In [ ]:
def supervisor_route(state: SupervisorState) -> str:
    """
    Si done=True -> 'end'
    Sino, mapea intent -> nombre del nodo agente
    """
    # TODO
    pass

print('Funcion condicional definida.')

## Paso 5 — TODO: Compilar el grafo

```python
builder = StateGraph(SupervisorState)
builder.add_node('supervisor_node', supervisor_node)
for name in ['hr_agent_node', 'it_agent_node', 'finance_agent_node', 'legal_agent_node']:
    builder.add_node(name, eval(name))

builder.set_entry_point('supervisor_node')

# Routing condicional: supervisor_node -> supervisor_route
# Mapping incluye 'end' -> END
builder.add_conditional_edges('supervisor_node', supervisor_route, {
    'hr_agent_node': 'hr_agent_node',
    'it_agent_node': 'it_agent_node',
    'finance_agent_node': 'finance_agent_node',
    'legal_agent_node': 'legal_agent_node',
    'end': END
})

# Cada agente vuelve al supervisor (no a END)
for name in ['hr_agent_node', 'it_agent_node', 'finance_agent_node', 'legal_agent_node']:
    builder.add_edge(name, 'supervisor_node')

graph = builder.compile()
```

In [ ]:
# TODO: construir el grafo supervisor
# Estructura:
# START -> supervisor_node
# supervisor_node -> conditional_edges con supervisor_route
# Cada agente_node -> supervisor_node (vuelven al supervisor)
# Compilar

graph = None  # reemplazar
print('Grafo compilado.')

In [ ]:
if graph:
    print(graph.get_graph().draw_mermaid())

## Paso 6 — Ejecutar con Langfuse

Probamos 4 queries. Cada ejecucion genera:

- **Trace** con tags `['m3l4', 'supervisor']`
- **Spans**: supervisor_node + agente_node (si aplica)
- **Datos en output**: intent, visited_agents, response

In [ ]:
queries = [
    'Como solicito mis dias de vacaciones?',
    'Mi VPN no conecta desde ayer',
    'Necesito ver mi factura del mes pasado',
    'ayuda'
]

if graph:
    for q in queries:
        lf = CallbackHandler()
        output = graph.invoke(
            {'query': q, 'intent': '', 'visited_agents': [], 'response': '', 'done': False},
            config={'callbacks': [lf], 'metadata': {'langfuse_tags': ['m3l4','supervisor']}}
        )
        print(f'Query:   {q[:45]}')
        print(f'Intent:  {output["intent"]}')
        print(f'Visited: {output["visited_agents"]}')
        print(f'Resp:    {output["response"][:60]}...')
        print()

In [ ]:
if graph:
    lf = CallbackHandler()
    r = graph.invoke({'query': 'No puedo ver mi factura', 'intent': '', 'visited_agents': [], 'response': '', 'done': False},
                     config={'callbacks': [lf]})
    assert r['intent'] == 'finance'
    assert 'FinanceAgent' in r['visited_agents']
    assert r['done'] == True
    assert len(r['response']) > 5
    print('Checks E10 OK')

## Errores comunes

| Error | Causa | Como detectarlo |
|---|---|---|
| Loop infinito | `done` nunca se marca `True` | El grafo no termina |
| `visited_agents` vacio | Los nodos agente no actualizan la lista | El historial no muestra que agente actuo |
| El supervisor siempre rutea a `general` | `route_query_v2` no detecta keywords | Revisar las reglas de routing |
| `supervisor_route` retorna nombre incorrecto | Desajuste entre string retornado y mapping | LangGraph lanza error de nodo desconocido |
| No hay edge de vuelta al supervisor | `add_edge(agent_node, END)` en vez de `add_edge(agent_node, 'supervisor_node')` | El agente no devuelve control al supervisor |

## Sintesis

### Que construiste

| Componente | Descripcion |
|---|---|
| `SupervisorState` | Estado con historial de agentes visitados y flag done |
| `supervisor_node()` | Nodo central que clasifica y controla el flujo |
| 4 nodos agente | Agentes especialistas que actualizan visited_agents |
| `supervisor_route()` | Routing condicional: sigue (agente) o termina (END) |
| Grafo con ciclo controlado | Agentes vuelven al supervisor, no a END |

### Diferencia con E09

| Aspecto | E09 (router simple) | E10 (supervisor) |
|---|---|---|
| Flujo | router -> agente -> END | supervisor -> agente -> supervisor -> ... -> END |
| Control | Un solo paso | Multiples rondas posibles |
| Historial | No | `visited_agents` evita loops |
| Estado | query, intent, response | + visited_agents, done |
| Spans en Langfuse | 2 por ejecucion | 2+ por ejecucion (si hay ciclo) |

### Relacion con otros ejercicios

| Ejercicio | Conexion con E10 |
|---|---|
| **E09** | Base del grafo con routing condicional |
| **E11** | Evaluar el supervisor completo con golden dataset |
| **E12** | Ciclo de mejora: el supervisor puede beneficiarse de mejoras en routing |